# Bài 2 — Trang 30 — Câu 2 (b): Lọc dữ liệu từ ngày 03/01/2018

**Võ Thiện Linh — MSSV: 6451071041**

Loại bỏ dữ liệu trước ngày 03/01/2018. Workbook có một dòng trống và các dòng thống kê tổng hợp ở cuối; notebook loại dòng trống, các dòng tổng hợp có ID không nguyên/trùng, rồi áp dụng mốc ngày.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
import pandas as pd
from IPython.display import display
columns = ['id', 'device_id', 'co_level', 'humidity', 'temperature', 'time']
candidates = [Path('tb_tracking.xlsx'), Path('LAB1') / 'Phan01' / '6451071041_LAB1_Bai2_Trang30' / 'tb_tracking.xlsx', Path('/content/tb_tracking.xlsx')]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy tb_tracking.xlsx. Thư mục đang chạy: {Path.cwd()}')
raw_df = pd.read_excel(data_path, header=None, names=columns)
rows_before_empty_drop = len(raw_df)
data = raw_df.dropna(how='all').copy()
empty_rows = rows_before_empty_drop - len(data)
data['id'] = pd.to_numeric(data['id'], errors='coerce')
data['time'] = pd.to_datetime(data['time'], errors='coerce')
invalid_rows = int(data[['id', 'time']].isna().any(axis=1).sum())
data = data.dropna(subset=['id', 'time']).copy()
integer_id = data['id'].mod(1).eq(0)
non_integer_id_rows = int((~integer_id).sum())
data = data.loc[integer_id].copy()
duplicate_id_rows = int(data['id'].duplicated(keep='first').sum())
data = data.drop_duplicates(subset='id', keep='first').copy()
data['id'] = data['id'].astype('int64')
cutoff = pd.Timestamp('2018-01-03')
before_cutoff_rows = int(data['time'].lt(cutoff).sum())
clean_df = data.loc[data['time'].ge(cutoff)].copy()
print(f'Dòng trống ban đầu: {empty_rows}')
print(f'Dòng thiếu ID hoặc thời gian hợp lệ đã bỏ: {invalid_rows}')
print(f'Dòng tổng hợp có ID không nguyên đã bỏ: {non_integer_id_rows}')
print(f'Dòng trùng ID đã bỏ: {duplicate_id_rows}')
print(f'Dòng trước ngày 03/01/2018 đã bỏ: {before_cutoff_rows}')
print(f'Số bản ghi sau lọc: {len(clean_df)}')
assert clean_df['id'].is_unique
assert clean_df['time'].ge(cutoff).all()
print('Năm dòng đầu sau lọc:')
display(clean_df.head())
print('Năm dòng cuối sau lọc:')
display(clean_df.tail())